# Spam Classification using Encoder LLMs with Linear Probing [5 points]
In this part, we will use encoder Large Language Models (LLMs) for spam classification. We will leverage the rich features of pre-trained LLMs without fine-tuning them. Instead, we will freeze the LLM weights and train a lightweight classifier head (MLP) on top for spam classification.

**Dataset:** Enron Spam Dataset

**Expected Performance (Best Model):** {Accuracy: >85%, F1: >85%, Precision: >85%, Recall: >82%}

1. Load the Enron Spam dataset. Use the train/val/test splits and tokenize the text using your pre-trained LLM’s tokenizer. Use your best judgement for the relevant input fields.

In [1]:
!pip install transformers datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system ==

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Loading Enron dataset
dataset = load_dataset("SetFit/enron_spam")
dataset = dataset.rename_column("label", "labels")

# Loading tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/176 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/101M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/6.27M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/31716 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [3]:
from datasets import DatasetDict
split_dataset = tokenized["train"].train_test_split(test_size=0.1, seed=42)
tokenized = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
    "test": tokenized["test"]
})


In [4]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(tokenized["train"], batch_size=16, shuffle=True, collate_fn=data_collator)
val_loader = DataLoader(tokenized["validation"], batch_size=32, shuffle=False, collate_fn=data_collator)
test_loader = DataLoader(tokenized["test"], batch_size=32, shuffle=False, collate_fn=data_collator)


2. Model Setup – Probing:

In [5]:
class ProbingClassifier(nn.Module):
    def __init__(self, base_model_name, hidden_dim=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        for param in self.encoder.parameters():
            param.requires_grad = False
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)
        )

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
            cls_embedding = outputs.last_hidden_state[:, 0, :]
        return self.head(cls_embedding)


   a. Load a pre-trained LLM (e.g., DistilBERT, BART-encoder) for sequence classification. Choose a lightweight encoder model that is amenable to your GPU size. Consider using DistilBERT, TinyBERT, MobileBERT, AlBERT, or others. **Specify the chosen LLM below.**

   b. Freeze all base model weights and attach a lightweight MLP (the classification head) that maps the model’s representations to binary labels. You may want to create a separate model class that defines these components and a forward function or use out of the box 🤗 classification wrappers.

   **Chosen Encoder LLM:** <span style='color:green'>distilbert-base-uncased</span>

In [6]:
# Evaluation function
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels.extend(batch["labels"].tolist())
            outputs = model(input_ids, attention_mask)
            preds.extend(torch.argmax(outputs, dim=1).cpu().tolist())
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)
    f1 = f1_score(labels, preds)
    print(f"Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}")

In [8]:
### ADD YOUR CODE HERE ###
# Loading pre-trained encoder LLM
model = ProbingClassifier(model_name).to(device)
optimizer = torch.optim.Adam(model.head.parameters(), lr=2e-4)
loss_fn = nn.CrossEntropyLoss()

def train(model, loader, val_loader, epochs=3):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1} Loss: {total_loss / len(loader):.4f}")
        evaluate(model, val_loader)

train(model, train_loader, val_loader, epochs=3)


100%|██████████| 1784/1784 [06:45<00:00,  4.40it/s]


Epoch 1 Loss: 0.1539
Acc: 0.9685 | Prec: 0.9677 | Rec: 0.9712 | F1: 0.9695


100%|██████████| 1784/1784 [06:44<00:00,  4.41it/s]


Epoch 2 Loss: 0.0866
Acc: 0.9738 | Prec: 0.9691 | Rec: 0.9804 | F1: 0.9747


100%|██████████| 1784/1784 [06:45<00:00,  4.40it/s]


Epoch 3 Loss: 0.0742
Acc: 0.9741 | Prec: 0.9743 | Rec: 0.9755 | F1: 0.9749


4. Evaluation and Analysis:

   a. Evaluate the model on the test set using accuracy, precision, recall, and F1-score.

In [9]:
print("Final Evaluation on Test dataset:")
evaluate(model, test_loader)


Final Evaluation on Test dataset:
Acc: 0.9810 | Prec: 0.9831 | Rec: 0.9792 | F1: 0.9811


   b. Select **two** encoder LLMs, repeat steps 2-4 for the second LLM, and compare and discuss any performance trends between the two models. **Specify the second chosen LLM below and report performance comparison.**

   **Second Chosen Encoder LLM:** <span style='color:green'> google/mobilebert-uncased </span>

In [10]:
# Training and evaluating MobileBERT probing model
mobilebert_model = ProbingClassifier("google/mobilebert-uncased", hidden_dim=512).to(device)
optimizer_mb = torch.optim.Adam(mobilebert_model.head.parameters(), lr=2e-4)

def train_mobilebert(model, loader, val_loader, epochs=3):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)

            optimizer_mb.zero_grad()
            loss.backward()
            optimizer_mb.step()
            total_loss += loss.item()
        print(f"[MobileBERT] Epoch {epoch+1} Loss: {total_loss / len(loader):.4f}")
        evaluate(model, val_loader)

train_mobilebert(mobilebert_model, train_loader, val_loader)

print("MobileBERT Final Evaluation on Test Set:")
evaluate(mobilebert_model, test_loader)

config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/147M [00:00<?, ?B/s]

  0%|          | 5/1784 [00:01<06:40,  4.44it/s]

model.safetensors:   0%|          | 0.00/147M [00:00<?, ?B/s]

100%|██████████| 1784/1784 [06:55<00:00,  4.29it/s]


[MobileBERT] Epoch 1 Loss: 81972.3619
Acc: 0.4987 | Prec: 0.7619 | Rec: 0.0392 | F1: 0.0745


100%|██████████| 1784/1784 [06:57<00:00,  4.28it/s]


[MobileBERT] Epoch 2 Loss: 0.6936
Acc: 0.5151 | Prec: 0.5151 | Rec: 1.0000 | F1: 0.6800


100%|██████████| 1784/1784 [06:56<00:00,  4.29it/s]


[MobileBERT] Epoch 3 Loss: 0.6930
Acc: 0.5151 | Prec: 0.5151 | Rec: 1.0000 | F1: 0.6800
MobileBERT Final Evaluation on Test Set:
Acc: 0.5040 | Prec: 0.5040 | Rec: 1.0000 | F1: 0.6702


   **Performance Comparison and Trend Discussion:**

   c. The best model is expected to attain {Accuracy: >85%, F1: >85%, Precision: >85%, Recall: >82%}. Report whether your best model achieves these metrics and discuss.

The best-performing model in our experiment, namely DistilBERT frozen encoder, and trainable classification head, achieved the following statistics on the test set:

Accuracy: 98.10%

Precision: 98.31%

Recall: 97.92%

F1 Score: 98.11%

The model has shown strong generalization such that it has high sensitivity (recall) and precision, which suggests that the embeddings from the frozen DistilBERT encoder are rich enough, separable and good for downstream tasks of binary classification; on the contrary, the performance metrics required for MobileBERT were not met, advocating the variability that probing performance varies even among different architectures of frozen LLMs.

   **Performance vs. Expected Metrics Discussion:**

5. References. Include details on all the resources used to complete this part.

https://huggingface.co/docs/transformers/en/model_doc/distilbert,https://huggingface.co/docs/transformers/en/model_doc/mobilebert,https://h2o.ai/wiki/probing-classifiers/,https://arxiv.org/abs/2102.12452,https://pytorch.org/tutorials/beginner/basics/intro.html,https://developers.google.com/machine-learning/resources/intro-llms.